# Lab 3: Task/Tool Agent via AgentCore Gateway

Expose three **synthetic** hospital operations as MCP tools through AgentCore Gateway,
exactly as the console build did — but created with boto3 here:
`check_appointment_status`, `check_refill_eligibility`, `stage_refill_request`.

`stage_refill_request` only **stages** (status `awaiting_approval`, `submitted=false`).
It must never submit. All data is synthetic.

### Step 1: Create the mock-hospital Lambda (synthetic data)

In [ ]:
import boto3, json, zipfile, io, time
import lab_helpers.utils as u

lambda_client = boto3.client("lambda", region_name=u.REGION)
account = u.get_aws_account_id()

LAMBDA_SRC = '''
import uuid
def get_tool_name(event, context):
    try:
        custom = context.client_context.custom
        full = custom.get("bedrockAgentCoreToolName", "")
        return full.split("___", 1)[1] if "___" in full else full
    except Exception:
        return event.get("_tool_name", "")
def lambda_handler(event, context):
    t = get_tool_name(event, context)
    if t == "check_appointment_status":
        return {"patient_reference": event.get("patient_reference","PATIENT-DEMO-001"),
                "appointment_status":"scheduled","department":"Gastroenterology",
                "appointment_reference":"APT-DEMO-1001"}
    if t == "check_refill_eligibility":
        return {"drug": event.get("drug","Unknown"),"eligible":True,
                "refills_remaining":2,"data_type":"synthetic"}
    if t == "stage_refill_request":
        return {"action":"submit_refill","drug":event.get("drug","Unknown"),
                "status":"awaiting_approval",
                "idempotency_key":event.get("idempotency_key",str(uuid.uuid4())),
                "submitted":False}
    return {"error":"Unknown tool","tool_name":t}
'''

# Zip in-memory
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as z:
    z.writestr("lambda_function.py", LAMBDA_SRC)
buf.seek(0)

# Lambda needs an execution role — reuse a simple basic-exec role.
lambda_role_arn = u._create_role(
    u.name("CareConnectMockToolsRole"), "lambda.amazonaws.com",
    {"Version":"2012-10-17","Statement":[{"Effect":"Allow",
       "Action":["logs:CreateLogGroup","logs:CreateLogStream","logs:PutLogEvents"],
       "Resource":"*"}]},
    u.name("CareConnectMockToolsPolicy"))

try:
    fn = lambda_client.create_function(
        FunctionName=u.MOCK_TOOLS_LAMBDA, Runtime="python3.12",
        Role=lambda_role_arn, Handler="lambda_function.lambda_handler",
        Code={"ZipFile": buf.read()}, Timeout=30)
    print("Created Lambda:", fn["FunctionArn"])
except lambda_client.exceptions.ResourceConflictException:
    fn = lambda_client.get_function(FunctionName=u.MOCK_TOOLS_LAMBDA)["Configuration"]
    print("Reusing Lambda:", fn["FunctionArn"])
lambda_arn = fn["FunctionArn"]

### Step 2: Create the AgentCore Gateway and Lambda target

Uses `bedrock-agentcore-control`. For this synthetic dev exercise we use NONE inbound
auth (AWS supports this for dev; use JWT/IAM for production).

In [ ]:
acc = boto3.client("bedrock-agentcore-control", region_name=u.REGION)

gw_role_arn = u._create_role(
    u.name("CareConnectGatewayRole"), "bedrock-agentcore.amazonaws.com",
    {"Version":"2012-10-17","Statement":[{"Effect":"Allow",
       "Action":["lambda:InvokeFunction"],"Resource": lambda_arn}]},
    u.name("CareConnectGatewayPolicy"))

# NOTE: adjust create_gateway args to your installed bedrock-agentcore-control schema.
gw = acc.create_gateway(
    name=u.GATEWAY_NAME,
    roleArn=gw_role_arn,
    protocolType="MCP",
    authorizerType="NONE",
    description="CareConnect synthetic hospital tools (SDK build).")
gateway_id = gw["gatewayId"]
gateway_url = gw["gatewayUrl"]
u.put_ssm_parameter(f"{u.SSM_PREFIX}/gateway_id", gateway_id)
u.put_ssm_parameter(f"{u.SSM_PREFIX}/gateway_url", gateway_url)
print("Gateway:", gateway_id, gateway_url)

In [ ]:
tool_schema = [
  {"name":"check_appointment_status",
   "description":"Check the status of a synthetic Riverside Health appointment.",
   "inputSchema":{"type":"object","properties":{
      "patient_reference":{"type":"string","description":"Synthetic patient reference."}},
      "required":["patient_reference"]}},
  {"name":"check_refill_eligibility",
   "description":"Check synthetic prescription refill eligibility. Does not submit a refill.",
   "inputSchema":{"type":"object","properties":{
      "drug":{"type":"string","description":"Synthetic medication name."}},
      "required":["drug"]}},
  {"name":"stage_refill_request",
   "description":"Prepare a synthetic refill request for human approval. Does not submit.",
   "inputSchema":{"type":"object","properties":{
      "drug":{"type":"string","description":"Synthetic medication name."},
      "idempotency_key":{"type":"string","description":"Unique id to prevent duplicates."}},
      "required":["drug","idempotency_key"]}},
]

acc.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name=u.name("careconnect-hospital-tools"),
    targetConfiguration={"mcp":{"lambda":{
        "lambdaArn": lambda_arn,
        "toolSchema":{"inlinePayload": tool_schema}}}},
    credentialProviderConfigurations=[{"credentialProviderType":"GATEWAY_IAM_ROLE"}])
print("Target attached.")

### Step 3: Task/Tool Agent connects to the Gateway over MCP

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

TASK_PROMPT = '''You are the CareConnect Task and Tool Agent for Riverside Health.
You may check appointment status, check refill eligibility, and stage a refill for
human approval. Never diagnose, never recommend medication or dosage, never claim a
staged refill was submitted. A staged refill stays awaiting_approval / submitted=false.
All data is synthetic.'''

mcp_client = MCPClient(lambda: streamablehttp_client(url=gateway_url))
with mcp_client:
    tools = mcp_client.list_tools_sync()
    agent = Agent(model=BedrockModel(model_id=u.MODEL_ID),
                  system_prompt=TASK_PROMPT, tools=tools)
    print(agent("Prepare a refill request for DemoMedication using key REFILL-DEMO-001."))

## Lab 3 complete ✅

Three synthetic tools discoverable/callable via Gateway; refills only stage.